# Lab 1 — the model boundary, in code

*Day 1 · after Module 1*

<a href="https://colab.research.google.com/github/MohammadYusif/llm-application-engineering/blob/main/labs/lab1-skeleton.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

*Runs in Colab with no API key and nothing installed locally. The first cell fetches the course and starts the gateway, a small local service that answers from rules rather than from a model — so every number below is real about this harness, and not a claim about any provider.*

Module 1 argued for one boundary between the application and the provider, a
pattern chosen on evidence, state you can bound, and reliability policy in a single
place. Every point it made is below, running against **Murshid** — the bilingual
citizen-services assistant this course builds on. Read a cell, run it, change a
value, run it again.

Murshid is the worked example. Your capstone is your own application, on a track
you choose, and the last section hands each of these points back to you for it.

## Setup

One cell. On Colab it fetches the course and starts the gateway; on your own
machine it finds the project you already have and checks the gateway is up.

In [1]:
import contextlib, os, pathlib, re, subprocess, sys, time, urllib.request, json

# pytest and ruff colour their output; those escapes render as noise once the
# notebook is published, so they come off here rather than per command.
ANSI = re.compile(chr(27) + r"\[[0-9;]*m")

REPO = "https://github.com/MohammadYusif/llm-application-engineering"
IN_COLAB = "google.colab" in sys.modules

# On Colab there is no checkout and no gateway, so fetch one and start one. The
# gateway is a local FastAPI app that answers from rules — no API key, no network
# calls out — which is the whole reason this course runs anywhere.
if IN_COLAB:
    root = pathlib.Path("/content/llm-application-engineering")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(root)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                        str(root / "murshid" / "requirements.lock")], check=True)
    os.chdir(root / "murshid")
else:
    for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (cand / "src" / "murshid").is_dir():
            os.chdir(cand); break
        if (cand / "murshid" / "src" / "murshid").is_dir():
            os.chdir(cand / "murshid"); break

sys.path.insert(0, "src")
os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("PYTHONPATH", "src")

# The application logs every routing decision and every model call. That is the
# point in production and noise in a notebook, so the default here is WARNING and
# the few sections where the log IS the lesson turn it back up themselves.
os.environ.setdefault("MURSHID_LOG_LEVEL", "WARNING")

@contextlib.contextmanager
def quiet():
    """Silence the application log inside a block that logs once per item.

    A loop over fifty corpus rows writes fifty validation warnings, and the
    report underneath them is the lesson. structlog freezes each module's logger
    on first use, so the level cannot be lowered after the fact — the writer is
    what gets muted instead.
    """
    import structlog
    levels = ("msg", "log", "debug", "info", "warn", "warning", "err", "error",
              "critical", "exception", "fatal", "failure")
    saved = {name: getattr(structlog.PrintLogger, name) for name in levels}
    for name in levels:
        setattr(structlog.PrintLogger, name, lambda self, message: None)
    try:
        yield
    finally:
        for name, fn in saved.items():
            setattr(structlog.PrintLogger, name, fn)

def run(*args, quiet_logs=True, may_fail=False):
    """Run a course command and print what it printed.

    quiet_logs drops the structured log lines so the boxed summary is readable;
    pass quiet_logs=False when the log IS the lesson.

    may_fail=True for the commands whose job is to exit non-zero: the gate when
    it blocks, and the uncalibrated judge. Everywhere else a non-zero exit stops
    the notebook, because a traceback printed into a page that still reports as
    executed is worse than no output at all.
    """
    out = subprocess.run([sys.executable, *args], capture_output=True, text=True,
                         encoding="utf-8", errors="replace")
    text = ANSI.sub("", out.stdout + out.stderr)
    if quiet_logs:
        # Structured logs come in two shapes — the console format on a laptop and
        # JSON lines in the container — so drop both, rather than whichever one
        # the machine that built this notebook happened to emit.
        def _is_log(line):
            if line.startswith("20") and "[" in line[:40]:
                return True
            return line.lstrip().startswith('{"') and (
                '"stage"' in line or '"event"' in line or '"logger"' in line)
        text = "\n".join(l for l in text.splitlines() if not _is_log(l))
    else:
        # The log is the lesson here, but not all of it: assistant_built and the
        # per-call llm_cost records are plumbing, and they are also the widest
        # lines on the page. Keep the retries, the failover and the refusals.
        NOISE = ("llm_cost", "assistant_built")
        text = "\n".join(l for l in text.splitlines()
                          if not any(n in l for n in NOISE))
    print(text.strip())
    if out.returncode and not may_fail:
        # A failing subprocess does not fail the notebook on its own, so say so
        # loudly. Without this a broken command is a traceback in the middle of a
        # page that still reports as executed cleanly.
        raise SystemExit(f"command failed with exit code {out.returncode}: {' '.join(args)}")
    return out.returncode

# The gateway is 127.0.0.1 on a laptop and `gateway` inside compose, so take it
# from the same environment variable the application routes through rather than
# hardcoding a host that is only right in one of the two places.
GATEWAY = os.environ.get("MURSHID_PRIMARY_BASE_URL", "http://127.0.0.1:8080/v1")
GATEWAY = GATEWAY.rsplit("/v1", 1)[0].rstrip("/")

# demo_v0.py is deliberately naive — hardcoded model, inline prompt, no timeout —
# but it does read OPENAI_BASE_URL, and its default is only right on a laptop.
# Point it at the same gateway as everything else so the lab works in both places.
os.environ.setdefault("OPENAI_BASE_URL", GATEWAY + "/v1")

def fault(payload):
    """Fault injection on the course gateway: the 429 storm and the outage drill."""
    req = urllib.request.Request(
        GATEWAY + "/admin/fault", method="POST",
        data=json.dumps(payload).encode(), headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.load(r)

def gateway_stats():
    with urllib.request.urlopen(GATEWAY + "/admin/stats", timeout=5) as r:
        return json.load(r)

def gateway_reset():
    """Clear the gateway's prompt cache, stats and faults."""
    req = urllib.request.Request(GATEWAY + "/admin/reset", method="POST", data=b"")
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.load(r)

def gateway_models(timeout=3):
    with urllib.request.urlopen(GATEWAY + "/healthz", timeout=timeout) as r:
        return json.load(r)["models"]

try:
    print("gateway:", gateway_models())
except Exception:
    if IN_COLAB:
        # Nothing is listening yet on a fresh runtime, so start it here. It runs
        # for the life of the notebook and needs no credentials.
        subprocess.Popen([sys.executable, "-m", "uvicorn", "app.main:app",
                          "--host", "127.0.0.1", "--port", "8080", "--log-level", "warning"],
                         cwd="infra/mockgw",
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        for _ in range(60):
            try:
                print("gateway:", gateway_models(timeout=2)); break
            except Exception:
                time.sleep(1)
        else:
            print("the course gateway did not come up — re-run this cell")
    else:
        print(f"gateway at {GATEWAY} is NOT answering — start it first:")
        print("   make gateway      (or)   docker compose up -d gateway")
print("cwd:", pathlib.Path.cwd())

gateway: ['course-flagship', 'course-small', 'course-anthropic', 'murshid-onprem']
cwd: /srv


## 1. LLM applications fail quietly

A web service that breaks returns a 500 and wakes someone. An LLM application that
breaks returns a fluent, confident, wrong answer, and nobody is woken — the fee is
plausible, the format is right, and the citizen acts on it.

The service directory is the ground truth here. `unsupported_amounts()` compares
every number in an answer against the numbers the directory actually contains, so
"quietly wrong" becomes something a machine can see.

In [2]:
from murshid.domain.directory import rendered_directory
from murshid.pipeline.groundedness import is_grounded, unsupported_amounts

directory = rendered_directory("en")
print("the directory the model is given:", len(directory), "characters")
for line in directory.splitlines():
    if "fee:" in line:
        print("  ", line.strip())

the directory the model is given: 5581 characters
   - fee: SAR 200 for each year of renewal
   - fee: SAR 200 issuance fee plus SAR 800 annual activity subscription
   - fee: No fee
   - fee: SAR 100
   - fee: SAR 50
   - fee: SAR 40 for each year of renewal
   - fee: SAR 150
   - fee: SAR 5 per square metre of built area
   - fee: SAR 300 per year for a retail activity
   - fee: No fee
   - fee: No fee


Now an answer that *reads* perfectly and is wrong in the one place that matters.
Nothing about its shape gives it away — only the comparison does.

In [3]:
plausible = ("Renewing a commercial registration costs SAR 750 per year "
             "and is issued within two working days.")

print("answer:", plausible)
print("amounts with no support in the directory:", unsupported_amounts(plausible, directory))
print("grounded?", is_grounded(plausible, directory))

answer: Renewing a commercial registration costs SAR 750 per year and is issued within two working days.
amounts with no support in the directory: {'750'}
grounded? False


That comparison is a **deterministic check**: no model, no judgement, no cost. It
is the cheapest thing in Module 5's harness and it catches the failure that a
demo never shows you. Note what it does *not* do — it says nothing about tone,
helpfulness or completeness. One check, one claim.

## 2. Four patterns, as an escalation ladder

A single call, a workflow, a router, a bounded agentic loop — in that order, and
you climb only when the traffic forces you to. Murshid is a **router**: it
classifies each message into an intent and hands it to the handler for that
intent, because its traffic really does split into questions and transactions.

Classification is itself a small model call, on the cheap route.

In [4]:
from murshid.app import build_client
from murshid.config import get_settings
from murshid.pipeline.router import IntentRouter

settings = get_settings()
router = IntentRouter(build_client(settings, settings.cheap_route))

for message in ["How much is a commercial licence renewal?",
                "Book me an appointment in Jeddah next Tuesday",
                "كم رسوم تجديد السجل التجاري؟",
                "My application REF-2291 has been stuck for three weeks"]:
    print(f"{router.classify(message):<10} <- {message}")

faq        <- How much is a commercial licence renewal?
service    <- Book me an appointment in Jeddah next Tuesday
faq        <- كم رسوم تجديد السجل التجاري؟
service    <- My application REF-2291 has been stuck for three weeks


Two of those are questions and two are transactions, in two languages, and the
router separates them before any expensive work happens. That is the whole
argument for the pattern: the cheap path stays cheap, and only the traffic that
needs tools pays for tools.

## 3. The model boundary

One interface. The application depends on `LLMClient`, and every provider is an
implementation of it. The request and the response are the course's own types, so
nothing above the boundary ever sees a provider's field names.

In [5]:
from murshid.llm.interfaces import LLMRequest, LLMResponse, Message

print("LLMRequest  :", ", ".join(LLMRequest.model_fields))
print("LLMResponse :", ", ".join(LLMResponse.model_fields))

LLMRequest  : messages, model_alias, temperature, max_tokens, tools, response_format, cache_prefix_messages
LLMResponse : text, tool_calls, finish_reason, model_id, usage, latency_ms, route


`usage`, `model_id` and `finish_reason` are on the response because control flow
depends on them: the cost meter needs the first two, and the tool loop branches on
the third. A boundary that returns only text throws that away and every layer
above has to guess.

The third implementation is the one you feel first — a fake, scripted in a line,
that needs no network and no key.

In [6]:
from murshid.llm.fake import FakeClient

fake = FakeClient(model_id="demo-model").script_text("SAR 200 per year.", tokens=(120, 9))
reply = fake.complete(LLMRequest(
    messages=[Message(role="user", content="Licence renewal fee?")], max_tokens=64))

print(type(reply).__name__, "->", reply.text)
print("model_id:", reply.model_id, "| finish:", reply.finish_reason,
      "| tokens in/out:", reply.usage.input_tokens, "/", reply.usage.output_tokens)

LLMResponse -> SAR 200 per year.
model_id: demo-model | finish: stop | tokens in/out: 120 / 9


Now the same request against a live route on the course gateway. Different
implementation, different model, **same two lines of calling code** — that is the
claim the boundary makes, and it is either true in your code or it is a diagram.

In [7]:
live = build_client(settings, settings.primary_route)
answer = live.complete(LLMRequest(
    messages=[Message(role="system", content="Answer only from this directory.\n" + directory),
              Message(role="user", content="How much does a commercial licence renewal cost?")],
    model_alias="murshid-flagship", max_tokens=200))

print(answer.text.strip())
print()
print("model_id:", answer.model_id, "| route:", answer.route,
      "| usage:", answer.usage.model_dump())

About Renewing a commercial registration (CR):
- Fee: SAR 200 for each year of renewal
- Processing time: Same working day once payment clears
- Documents required: Valid national ID or Iqama; Current municipality licence; Zakat certificate for the last closed year
- Steps:
  1. Sign in to the portal with your national ID
  2. Open Business Services and choose Renew Commercial Registration
  3. Confirm the activity and the renewal period
  4. Pay through SADAD and download the renewed certificate
If you need more help you can contact Any Digital Government Services Authority centre, or the 24/7 line 199.

model_id: course-flagship | route: primary | usage: {'input_tokens': 1150, 'output_tokens': 138, 'cached_input_tokens': 0}


Two things worth noticing before moving on. The answer is **grounded in the
directory that arrived in the prompt** — take the directory away and the same
question cannot be answered. And `usage` came back populated, which is what makes
Module 6's cost meter possible at all.

In [8]:
print("grounded?", is_grounded(answer.text, directory))

grounded? True


The architecture test is what keeps the boundary honest. It fails the build if
`openai` or `anthropic` is imported anywhere except the two adapter files.

In [9]:
run("-m", "pytest", "tests/test_architecture.py")

......                                                                   [100%]
6 passed in 0.11s


0

## 4. State, streaming, and the sync/batch line

The model is stateless. Every turn resends the whole history, so "memory" is a
decision your application makes and pays for. Murshid's default is a **window**:
keep the last N turns, drop the rest.

In [10]:
from murshid.domain.session import ConversationState

state = ConversationState(max_turns=2)   # small, so the effect is visible
for i in range(1, 4):
    state.add_user(f"question {i}")
    state.add_assistant(f"answer {i}")

print("turns kept:", [m.content for m in state.turns])
print("is 'question 1' still in the window?",
      any(m.content == "question 1" for m in state.turns))

turns kept: ['question 2', 'answer 2', 'question 3', 'answer 3']
is 'question 1' still in the window? False


Turn 1 is gone, on cue. Nothing is broken — a window is a *decision with a cost
curve*, and the alternative, unbounded history, is a bill that grows every turn
until the context overflows for your most engaged users first.

What `messages()` sends is the system prompt plus that window, rebuilt each turn.

In [11]:
sent = state.messages(system="You are Murshid.")
for m in sent:
    print(f"{m.role:<9} {m.content[:60]}")
print()
print("messages on the wire this turn:", len(sent))

system    You are Murshid.
user      question 2
assistant answer 2
user      question 3
assistant answer 3

messages on the wire this turn: 5


Streaming changes the *perceived* latency, not the total. The number that matters
is time-to-first-token: the wait before anything appears on screen.

In [12]:
import time

start = time.perf_counter()
ttft, chunks = None, 0
for chunk in live.stream(LLMRequest(
        messages=[Message(role="system", content=directory),
                  Message(role="user", content="What documents do I need to renew a licence?")],
        model_alias="murshid-flagship", max_tokens=160)):
    if chunk.delta and ttft is None:
        ttft = (time.perf_counter() - start) * 1000
    chunks += 1
total = (time.perf_counter() - start) * 1000

print(f"chunks: {chunks}   time to first token: {ttft:.0f} ms   total: {total:.0f} ms")
print(f"the reader waits {ttft:.0f} ms instead of {total:.0f} ms — the same work, felt differently")

chunks: 99   time to first token: 62 ms   total: 132 ms
the reader waits 62 ms instead of 132 ms — the same work, felt differently


## 5. Reliability lives at the boundary, once

Timeouts, retries with jitter, and failover belong in one place. Put them in each
call site and you get four different policies, three of which are wrong, and a
retry storm the first time a provider is slow rather than down.

`FakeClient` can script a rate limit, so the policy can be tested without waiting
for a real one.

In [13]:
from murshid.llm.resilient import ResilientClient

flaky = FakeClient(model_id="primary-model").script_rate_limit(times=2)
flaky.script_text("Answered after two 429s.", tokens=(80, 12))

# The chain is a list of named hops; sleep is injected so the lab does not
# actually wait out the backoff.
client = ResilientClient([("primary", flaky)], max_attempts=3, sleep=lambda _: None)
out = client.complete(LLMRequest(messages=[Message(role="user", content="hello")], max_tokens=64))

print("text:", out.text)
print("calls the primary actually took:", flaky.call_count)

2026-09-06T15:28:28.002812Z [warning  ] llm_retry                      attempt=1 delay_s=2.0 hop=primary retry_after=2.0 status=429


2026-09-06T15:28:28.003387Z [warning  ] llm_retry                      attempt=2 delay_s=2.0 hop=primary retry_after=2.0 status=429


text: Answered after two 429s.
calls the primary actually took: 3


Two 429s, one answer, and the calling code never knew. Now the case retries cannot
fix — the primary is *down*, not busy — where the second route earns its keep.

In [14]:
from murshid.llm.interfaces import LLMError

dead = FakeClient(model_id="primary-model")
dead.script_error(LLMError("connection refused"), times=3)
spare = FakeClient(model_id="fallback-model").script_text("Served by the fallback route.")

client = ResilientClient([("primary", dead), ("on_prem", spare)],
                         max_attempts=2, sleep=lambda _: None)
out = client.complete(LLMRequest(messages=[Message(role="user", content="hello")], max_tokens=64))

print("text:", out.text)
print("answered by:", out.model_id)

2026-09-06T15:28:28.009034Z [warning  ] llm_not_retryable              error=LLMError hop=primary status=None


2026-09-06T15:28:28.009452Z [warning  ] fallback_served                hop=on_prem model_id=fallback-model


text: Served by the fallback route.
answered by: fallback-model


And when every hop is exhausted, the application still owes the citizen a sentence
rather than a stack trace. A degraded reply is a product decision, made in advance.

In [15]:
from murshid.llm.resilient import degraded_response

for language in ("en", "ar"):
    print(language, "->", degraded_response(language).text)

en -> I can't answer right now because of a temporary fault. You can check the service directory or call the service centre.
ar -> أعتذر، لا أستطيع الإجابة في هذه اللحظة بسبب عُطل مؤقت. يمكنك مراجعة دليل الخدمات أو الاتصال بمركز الخدمة.


## 6. Common mistakes

Module 1 listed six. Three of them are visible in the cells above:

- **calling the SDK from the handler** — then a provider change is a rewrite, and
  the architecture test above is what stops it happening by accident;
- **unbounded history** — the window is a decision; make it deliberately and
  measure what it costs;
- **retry logic per call site** — one policy, at the boundary, tested against a
  scripted 429 rather than hoped for.

The other three are cheaper to fix now than in week three: no `usage` on the
response, no timeout on the client, and a demo whose numbers nobody can reproduce.

## Your turn — on your own project

Everything above ran against Murshid. Your capstone is **your** application, on the
track you pick, and it needs the same four things from this module. Start them now
rather than on Day 4:

1. **Name the shape.** Which of the four patterns does your traffic want — a single
   call, a workflow, a router, or a bounded agentic loop? One paragraph justifying
   it against the traffic mix you expect. That paragraph is your first ADR.
2. **Draw the boundary before you write an adapter.** One interface, a normalised
   request and response carrying `model_id`, `usage` and a finish reason. The shape
   in `src/murshid/llm/interfaces.py` is there to copy.
3. **Decide your state strategy** and say what it costs. Windowed is the safe
   default; if you pick summarisation, you owe the eval cases showing a long
   conversation still remembers what matters.
4. **Write your own context budget** — your directory, your tool schemas, your
   history. Measure the numbers rather than guessing them.

**Next:** [Module 2 — APIs and open weights](../modules/m2-apis-and-open-weights.qmd),
then [Lab 2](lab2-two-providers.ipynb).